Ce notebook a pour objectif d’améliorer les performances du modèle en optimisant ses hyperparamètres.

L’optimisation est réalisée sur un modèle de type Random Forest, qui est déjà le modèle le plus performant parmi ceux testés précédemment.

 1. Chargement des données

Le dataset est chargé directement depuis le fichier prétraité (train.csv), afin de garantir une continuité avec les étapes précédentes.

Les mêmes transformations sont appliquées pour garder un pipeline cohérent.

 2. Feature Engineering

Les mêmes variables dérivées sont recréées :

TotalSF : surface totale de la maison

HouseAge : ancienneté de la maison

QualityScore : score combinant qualité et condition générale

Ces variables permettent d’améliorer la représentation du problème pour le modèle.

 3. Sélection des variables

Les features utilisées sont identiques aux notebooks précédents :

GrLivArea

OverallQual

GarageCars

TotalBsmtSF

YearBuilt

TotalSF

HouseAge

QualityScore

La cible reste :

SalePrice

 4. Préparation des données

Les données sont nettoyées en supprimant les valeurs manquantes, puis divisées en :

80% entraînement

20% test

Cette séparation permet de tester la capacité de généralisation du modèle.

 5. Optimisation avec GridSearchCV

Une recherche d’hyperparamètres est effectuée sur le modèle Random Forest.

Les paramètres testés sont :

n_estimators : nombre d’arbres (100, 200)

max_depth : profondeur maximale des arbres (10, 20, None)

min_samples_split : nombre minimum d’échantillons pour diviser un nœud (2, 5)

 6. Validation croisée

La validation croisée (CV=3) est utilisée pour :

éviter le surapprentissage

tester la robustesse du modèle

sélectionner les meilleurs hyperparamètres

La métrique utilisée pour l’évaluation est le R² score.

 7. Meilleur modèle

Le meilleur modèle est automatiquement sélectionné à partir de GridSearchCV.

Ses performances sont ensuite évaluées sur le jeu de test.

 8. Évaluation finale

Le modèle optimisé est évalué avec :

MAE : erreur absolue moyenne

RMSE : erreur quadratique moyenne

R² : capacité d’explication du modèle

In [2]:
import pandas as pd
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==============================
# LOAD DATASET (STABLE VERSION)
# ==============================

df = pd.read_csv("data/raw/train.csv")

# ==============================
# FEATURE ENGINEERING
# ==============================

df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["QualityScore"] = df["OverallQual"] * df["OverallCond"]

# ==============================
# FEATURES
# ==============================

features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt",
    "TotalSF",
    "HouseAge",
    "QualityScore"
]

target = "SalePrice"

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ==============================
# SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ==============================
# GRID SEARCH
# ==============================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)

# ==============================
# EVALUATION
# ==============================

preds = best_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds)
rmse = rmse ** 0.5
r2 = r2_score(y_test, preds)

print("\n📊 Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

Best parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}

📊 Results:
MAE: 18195.08507043931
RMSE: 29298.55592299648
R2: 0.8880875003244812
